# Leak Detection Demo

本 Notebook 演示如何：
- 预览示例数据
- 查看与注册因子
- 运行泄露检测（全市场 as-of 重放）
- 生成并查看报告摘要

建议在项目根目录（含 `leak_detection` 与 `factors`）下运行。

In [6]:
import os
from pathlib import Path
import sys

# Get the current working directory (for Jupyter notebooks, this would be where the notebook is located)
notebook_path = Path(os.getcwd())

# Traverse up to the project root directory (assuming your project structure has the root two levels up from the notebook)
PROJECT_ROOT = notebook_path.parents[0]

# Add the project root to the system path for module import
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Your imports
from leak_detection.data import load_sample_factors, describe_dataset
from leak_detection.detector import detect_leaks
from leak_detection.reporting import to_summary_frame, export_all
from factors import get_registered_factors

PROJECT_ROOT

PosixPath('/Users/ods/Desktop/汇丰生活/社团/AFT/leak-detector-starter')

## 1. 载入与预览数据

In [7]:
df = load_sample_factors()
descriptor = describe_dataset(df)
display(df.head())
descriptor

,date,symbol,open,high,low,close,volume,ret_1d
0,2021-01-04,AAA,99.396665,99.641596,99.178606,99.463282,6705,NaN
1,2021-01-05,AAA,98.455335,98.671934,98.018850,98.423298,14778,-0.010456
2,2021-01-06,AAA,99.290025,100.639563,98.076601,99.173749,19439,0.007625
3,2021-01-07,AAA,100.256722,101.093977,99.685067,100.114314,18012,0.009484
4,2021-01-08,AAA,98.319034,98.669906,96.655730,98.163279,9111,-0.019488


DatasetDescriptor(rows=180, symbols=['AAA', 'BBB', 'CCC'], start_date='2021-01-04', end_date='2021-03-26', factor_columns=[], target_columns=[])

## 2. 查看注册的因子

In [8]:
factors = get_registered_factors()
list(factors.keys())

['factor_momentum_ema',
 'factor_volume_mean5',
 'factor_future_return',
 'factor_ret_1d_lag1',
 'factor_sma_5',
 'factor_volatility_10',
 'factor_rsi_14',
 'factor_obv_lag1',
 'factor_future_close',
 'factor_future_volume']

## 3. 运行泄露检测（全市场 as-of 重放）

In [9]:
result = detect_leaks(
    df,
    factor_definitions=factors,
    entity_col="symbol",
    time_col="date",
    tolerance=1e-9,
)
summary = to_summary_frame(result)
display(summary)
print(result.notes or "")

,factor,signal_type,confidence,extra
0,factor_future_close,forward_replay,1.0,"{'total_checked': 177, 'total_mismatch': 177, ..."
1,factor_future_close,consistency_check,1.0,"{'train_rate': 1.0, 'valid_rate': 1.0, 'note':..."
2,factor_future_return,forward_replay,1.0,"{'total_checked': 177, 'total_mismatch': 177, ..."
3,factor_future_return,consistency_check,1.0,"{'train_rate': 1.0, 'valid_rate': 1.0, 'note':..."
4,factor_future_volume,forward_replay,1.0,"{'total_checked': 177, 'total_mismatch': 177, ..."
5,factor_future_volume,consistency_check,1.0,"{'train_rate': 1.0, 'valid_rate': 1.0, 'note':..."


## 4. 导出报告文件（CSV/JSON/Markdown）

In [10]:
output_dir = PROJECT_ROOT / "outputs/notebook"
artefacts = export_all(result, output_dir)
artefacts

{'summary': PosixPath('/Users/ods/Desktop/汇丰生活/社团/AFT/leak-detector-starter/outputs/notebook/summary.csv'),
 'json': PosixPath('/Users/ods/Desktop/汇丰生活/社团/AFT/leak-detector-starter/outputs/notebook/violations.json'),
 'markdown': PosixPath('/Users/ods/Desktop/汇丰生活/社团/AFT/leak-detector-starter/outputs/notebook/report.md')}

## 5. 进一步分析（可选）
- 仅查看 forward_replay 信号
- 检视单个因子的明细样例

In [11]:
forward_only = summary[summary['signal_type'] == 'forward_replay'].copy()
forward_only.sort_values('confidence', ascending=False).reset_index(drop=True)

,factor,signal_type,confidence,extra
0,factor_future_close,forward_replay,1.0,"{'total_checked': 177, 'total_mismatch': 177, ..."
1,factor_future_return,forward_replay,1.0,"{'total_checked': 177, 'total_mismatch': 177, ..."
2,factor_future_volume,forward_replay,1.0,"{'total_checked': 177, 'total_mismatch': 177, ..."
